# 02. Text Preprocessing & Tokenization Pipeline
## Text Summarization Using Pre-trained Models

**Project:** Academic NLP Summarization Pipeline  
**Objective:** Build a robust, non-destructive cleaning pipeline for news articles and human reference summaries, format data for encoder-decoder Transformer architectures, and verify subword tokenization.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from transformers import AutoTokenizer

from config import RAW_TEST_FILE, PRIMARY_MODEL_NAME, ALTERNATIVE_MODEL_NAME
from src.preprocessing import clean_text, clean_highlights, get_text_length_info, tokenize_for_model, load_processed_split
print("Preprocessing modules loaded successfully.")

## 1. Raw Text Cleaning Demonstrations
News articles frequently contain syndicated wire prefixes (e.g., `(CNN) --`), irregular whitespace, and spacing before punctuation. We demonstrate non-destructive cleaning that preserves grammatical casing and punctuation.

In [ ]:
raw_sample = pd.read_csv(RAW_TEST_FILE).iloc[0]
print("--- RAW ARTICLE SNIPPET ---")
print(raw_sample['article'][:300])

cleaned_article = clean_text(raw_sample['article'])
print("\n--- CLEANED ARTICLE SNIPPET ---")
print(cleaned_article[:300])

print("\n--- RAW HIGHLIGHTS ---")
print(raw_sample['highlights'])

cleaned_summary = clean_highlights(raw_sample['highlights'])
print("\n--- CLEANED HIGHLIGHTS ---")
print(cleaned_summary)

## 2. Model-Specific Tokenization Verification
We verify tokenization for **BART** (Byte-Pair Encoding, 1,024 token limit) and **T5** (SentencePiece, 512 token limit with `summarize:` prefix).

In [ ]:
bart_tokenizer = AutoTokenizer.from_pretrained(PRIMARY_MODEL_NAME)
bart_tokens = tokenize_for_model(cleaned_article, bart_tokenizer, max_length=1024)
print(f"BART input_ids shape:      {bart_tokens['input_ids'].shape}")
print(f"BART attention_mask shape: {bart_tokens['attention_mask'].shape}")

t5_tokenizer = AutoTokenizer.from_pretrained(ALTERNATIVE_MODEL_NAME)
t5_tokens = tokenize_for_model(cleaned_article, t5_tokenizer, max_length=512)
print(f"T5 input_ids shape:        {t5_tokens['input_ids'].shape}")
print(f"T5 decoded prefix check:   {t5_tokenizer.decode(t5_tokens['input_ids'][0][:10])}")

## 3. Processed Dataset Split Audit
We inspect the cleaned records saved to `data/processed/`.

In [ ]:
clean_test_df = load_processed_split("test")
print(f"Processed Test Records: {len(clean_test_df)}")
print("Average article word count:", clean_test_df['article_word_count'].mean())
print("Average summary word count:", clean_test_df['summary_word_count'].mean())
clean_test_df[['id', 'article_word_count', 'summary_word_count']].head()